In [8]:
!pip install -q -U langchain langchain-google-genai

In [9]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.3
)

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! Yes, we offer eggless versions for all our cakes. Our Plum cake is Rs 500, Chocolate cake is Rs 650, and cupcakes are Rs 80 each. Please note that we require 24 hours' notice for all orders. How can I help you today?
Your name is Ravi! How can I help you with your order today?


In [11]:
history

[('human', "Hi, I'm Ravi. Do you make eggless cakes?"),
 ('ai',
  "Hi Ravi! Yes, we offer eggless versions for all our cakes. Our Plum cake is Rs 500, Chocolate cake is Rs 650, and cupcakes are Rs 80 each. Please note that we require 24 hours' notice for all orders. How can I help you today?"),
 ('human', "What's my name?"),
 ('ai', 'Your name is Ravi! How can I help you with your order today?')]

In [12]:
import csv
def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])

    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

In [13]:
run_tests()

PASS | How much is a plum cake? → Hi there! A delicious plum cake is Rs 500. We also offer eggless versi
PASS | Do you make eggless cakes? → Hello! Yes, we offer eggless versions for all our cakes. Our plum cake
PASS | Can I get a cake today evening? → Hello! I'm sorry, but we require 24 hours' notice for all orders. I'll
PASS | Do you deliver to Kakinada? → Hello! Thanks for reaching out to Sweet Crumbs. I'll check with the ow
PASS | What is the price of 6 cupcakes? → Hello! A cupcake is Rs 80, so 6 cupcakes will be Rs 480. Please note w

5/5 passed (100%)


In [7]:
from langchain.agents import create_agent
agent_order = create_agent(model=llm, tools=[], system_prompt=SYSTEM)  # tools come tomorrow
result_order = agent_order.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result_order["messages"][-1].content)

[{'type': 'text', 'text': "Hello! Yes, we offer eggless versions for all our cakes. Our plum cake is Rs 500, chocolate cake is Rs 650, and cupcakes are Rs 80 each. Please note that we require 24 hours' notice for all orders. How can I help you today?", 'extras': {'signature': 'EqkBCqYBAWkUfRMWBhbJHXobQuRUhaxKZ1zISn2cGXOFbnu6n/pOZSLluj2woOUMlZ6hNMhejNwczGo9ZvJx9uqqRbyiNUT0zX41yNDD70IS3z1oGokgob/6mRw7Yy3vzHbnjewsAft7IvW4FKl6MT2wi4lROJThsSHihHfkSwqdEO1GinFBUC9EMqoauXlGuLJbXBgHnEIwnuDpNzu3Qje9HKA4+XDdhMemow=='}}]
